# Notebook 03 — PINN Training

Trains a **Deep Ensemble of 5 Physics-Informed Neural Networks** to estimate:
- `H_sys(t)` — system inertia constant (MWs/MVA)
- `D(t)` — damping coefficient (MW/Hz)
- `RoCoF(t)` — rate of change of frequency (Hz/s)

Physics constraint enforced: **swing equation**
$$2 \cdot H(t) \cdot \frac{df}{dt} = \frac{\Delta P(t)}{P_{total}(t) \cdot f_0}$$

**Loss:**
$$\mathcal{L} = \mathcal{L}_{data} + \lambda_{phys} \cdot \mathcal{L}_{swing} + \lambda_{smooth} \cdot \mathcal{L}_{TV}$$

---
**Reads:** `../data/processed/de_inertia_15min.csv`  
**Writes:** `../checkpoints/member_{i}.pt`, `../checkpoints/scaler.pkl`, `../checkpoints/training_log.csv`

## 0. Imports & reproducibility

In [ ]:
import sys
import random
import pickle
import warnings
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (16, 4)

# Project root on path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.models.pinn import GridInertiaPINN, DeepEnsemble, build_time_features, INPUT_DIM
from src.losses import PINNLoss

# ── Reproducibility ──────────────────────────────────────────────────────────
GLOBAL_SEED = 42

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(GLOBAL_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 1. Config

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path("../data/processed")
CKPT_DIR  = Path("../checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Data ──────────────────────────────────────────────────────────────────────
WINDOW_SIZE  = 96        # timesteps per sample  (96 × 15min = 24h)
STRIDE       = 24        # sliding window stride (6h between samples)
TRAIN_FRAC   = 0.75
VAL_FRAC     = 0.15
# TEST_FRAC  = 0.10      # remainder

# ── Model ─────────────────────────────────────────────────────────────────────
HIDDEN_DIM   = 128
N_LAYERS     = 6
DROPOUT_P    = 0.15
N_MEMBERS    = 5         # ensemble size

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 80
BATCH_SIZE   = 64        # number of windows per batch
LR           = 3e-4
WEIGHT_DECAY = 1e-5

# ReduceLROnPlateau
LR_PATIENCE  = 8         # epochs without improvement before LR drop
LR_FACTOR    = 0.5
LR_MIN       = 1e-6

# Early stopping
ES_PATIENCE  = 15

# ── Loss weights ──────────────────────────────────────────────────────────────
LAMBDA_PHYS    = 1.0
LAMBDA_SMOOTH  = 0.01
WARMUP_EPOCHS  = 10      # ramp λ_phys from 0 → LAMBDA_PHYS over this many epochs

print("Config loaded.")

## 2. Load & inspect data

In [ ]:
df = pd.read_csv(
    DATA_DIR / "de_inertia_15min.csv",
    index_col="utc_timestamp",
    parse_dates=True
).sort_index()

# Drop rows where the supervision target is NaN
df = df.dropna(subset=["H_sys"])

print(f"Shape      : {df.shape}")
print(f"Date range : {df.index.min()}  →  {df.index.max()}")
print(f"Columns    : {list(df.columns)}")
df.describe().round(3)

## 3. Feature engineering

In [ ]:
# ── Power signal features (9) ─────────────────────────────────────────────────
POWER_COLS = [
    "P_load",
    "P_solar",
    "P_wind_onshore",
    "P_wind_offshore",
    "P_thermal",
    "P_hydro",
    "P_total",
    "renewables_fraction",
    "RoCoF_proxy",          # used as input feature AND soft supervision signal
]

# ── Derived imbalance ─────────────────────────────────────────────────────────
df["delta_P"] = df["P_load"] - (
    df["P_solar"] + df["P_wind_onshore"] + df["P_wind_offshore"]
    + df["P_thermal"] + df["P_hydro"]
)

# ── Time features (5) ─────────────────────────────────────────────────────────
YEAR_MIN   = float(df.index.year.min())
YEAR_RANGE = float(df.index.year.max() - df.index.year.min()) or 1.0

two_pi = 2 * np.pi
df["sin_hour"]  = np.sin(two_pi * df.index.hour / 24)
df["cos_hour"]  = np.cos(two_pi * df.index.hour / 24)
df["sin_month"] = np.sin(two_pi * (df.index.month - 1) / 12)
df["cos_month"] = np.cos(two_pi * (df.index.month - 1) / 12)
df["year_trend"]= (df.index.year - YEAR_MIN) / YEAR_RANGE

TIME_COLS = ["sin_hour", "cos_hour", "sin_month", "cos_month", "year_trend"]
FEATURE_COLS = POWER_COLS + TIME_COLS   # 14 features  == INPUT_DIM

assert len(FEATURE_COLS) == INPUT_DIM, (
    f"Feature count mismatch: {len(FEATURE_COLS)} != {INPUT_DIM}"
)

print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

## 4. Normalisation

In [ ]:
# Train/val/test split on raw index BEFORE fitting scaler
n        = len(df)
n_train  = int(n * TRAIN_FRAC)
n_val    = int(n * VAL_FRAC)

df_train = df.iloc[:n_train]
df_val   = df.iloc[n_train : n_train + n_val]
df_test  = df.iloc[n_train + n_val :]

print(f"Train : {len(df_train):>7,} rows  ({df_train.index.min().date()} → {df_train.index.max().date()})")
print(f"Val   : {len(df_val):>7,} rows  ({df_val.index.min().date()} → {df_val.index.max().date()})")
print(f"Test  : {len(df_test):>7,} rows  ({df_test.index.min().date()} → {df_test.index.max().date()})")

# Fit scaler ONLY on train split — scale power features, not time features
scaler = StandardScaler()
scaler.fit(df_train[POWER_COLS].fillna(0))

# Save scaler for inference
with open(CKPT_DIR / "scaler.pkl", "wb") as f:
    pickle.dump({"scaler": scaler, "power_cols": POWER_COLS,
                 "time_cols": TIME_COLS, "year_min": YEAR_MIN,
                 "year_range": YEAR_RANGE}, f)
print("Scaler saved.")

def scale_df(df_split: pd.DataFrame) -> np.ndarray:
    """Return normalised feature matrix (N, 14)."""
    X_power = scaler.transform(df_split[POWER_COLS].fillna(0))
    X_time  = df_split[TIME_COLS].values
    return np.concatenate([X_power, X_time], axis=1).astype(np.float32)

X_train = scale_df(df_train);  y_train = df_train["H_sys"].values.astype(np.float32)
X_val   = scale_df(df_val);    y_val   = df_val["H_sys"].values.astype(np.float32)
X_test  = scale_df(df_test);   y_test  = df_test["H_sys"].values.astype(np.float32)

dP_train = df_train["delta_P"].values.astype(np.float32)
dP_val   = df_val["delta_P"].values.astype(np.float32)
dP_test  = df_test["delta_P"].values.astype(np.float32)

Pt_train = df_train["P_total"].values.astype(np.float32)
Pt_val   = df_val["P_total"].values.astype(np.float32)
Pt_test  = df_test["P_total"].values.astype(np.float32)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

## 5. Dataset & DataLoader

In [ ]:
class GridInertiaDataset(Dataset):
    """Sliding-window dataset.

    Each sample is a contiguous window of `window_size` timesteps.
    Windows are extracted with `stride` step, then shuffled.
    Returning full windows preserves temporal order within each sample,
    which is required for SmoothnessLoss to make physical sense.
    """

    def __init__(
        self,
        X:           np.ndarray,   # (N, 14)
        y_H:         np.ndarray,   # (N,)  H_sys target
        delta_P:     np.ndarray,   # (N,)  power imbalance
        P_total:     np.ndarray,   # (N,)
        window_size: int = WINDOW_SIZE,
        stride:      int = STRIDE,
    ):
        self.X        = X
        self.y_H      = y_H
        self.delta_P  = delta_P
        self.P_total  = P_total
        self.W        = window_size

        # All valid start indices
        self.starts = list(range(0, len(X) - window_size + 1, stride))

    def __len__(self) -> int:
        return len(self.starts)

    def __getitem__(self, idx: int) -> dict:
        s = self.starts[idx]
        e = s + self.W
        return {
            "X":       torch.from_numpy(self.X[s:e]),          # (W, 14)
            "H_sys":   torch.from_numpy(self.y_H[s:e]),        # (W,)
            "delta_P": torch.from_numpy(self.delta_P[s:e]),    # (W,)
            "P_total": torch.from_numpy(self.P_total[s:e]),    # (W,)
        }


def make_loader(X, y_H, delta_P, P_total, shuffle: bool) -> DataLoader:
    ds = GridInertiaDataset(X, y_H, delta_P, P_total)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == "cuda"))


train_loader = make_loader(X_train, y_train, dP_train, Pt_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   dP_val,   Pt_val,   shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

# Inspect one batch
sample = next(iter(train_loader))
print(f"Batch X shape : {sample['X'].shape}   → (B, W, 14)")
print(f"Batch H shape : {sample['H_sys'].shape} → (B, W)")

## 6. Training utilities

In [ ]:
def run_epoch(
    model:     GridInertiaPINN,
    loader:    DataLoader,
    criterion: PINNLoss,
    optimizer: torch.optim.Optimizer | None,
    training:  bool,
) -> dict:
    """One full pass over loader. Returns mean loss breakdown."""
    model.train(training)
    totals = {"loss_total": 0., "loss_data": 0.,
              "loss_swing": 0., "loss_smooth": 0.}
    n_batches = 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            X       = batch["X"].to(DEVICE)                # (B, W, 14)
            H_tgt   = batch["H_sys"].to(DEVICE)            # (B, W)
            delta_P = batch["delta_P"].to(DEVICE)          # (B, W)
            P_total = batch["P_total"].to(DEVICE)          # (B, W)

            B, W, F = X.shape

            # Flatten window dim into batch for the model  → (B*W, 14)
            X_flat  = X.view(B * W, F)
            Ht_flat = H_tgt.view(B * W)
            dP_flat = delta_P.view(B * W)
            Pt_flat = P_total.view(B * W)

            H_pred, D_pred, RoCoF_pred = model(X_flat)

            # NaN mask — skip timesteps where H_sys target is NaN
            mask = ~torch.isnan(Ht_flat)

            loss, bd = criterion(
                H_pred     = H_pred,
                D_pred     = D_pred,
                RoCoF_pred = RoCoF_pred,
                H_target   = Ht_flat,
                delta_P    = dP_flat,
                P_total    = Pt_flat,
                mask       = mask,
            )

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            for k in totals:
                totals[k] += bd[k]
            n_batches += 1

    return {k: v / n_batches for k, v in totals.items()}


class EarlyStopping:
    """Stop training when val loss stops improving."""

    def __init__(self, patience: int = ES_PATIENCE, min_delta: float = 1e-5):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_loss   = float("inf")
        self.best_state  = None
        self.counter     = 0
        self.should_stop = False

    def step(self, val_loss: float, model: nn.Module):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.best_state = deepcopy(model.state_dict())
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

    def restore_best(self, model: nn.Module):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


print("Training utilities defined.")

## 7. Train all ensemble members

In [ ]:
all_logs = []   # one list of dicts per member

for member_id in range(N_MEMBERS):
    seed = GLOBAL_SEED + member_id * 100
    set_seed(seed)

    print(f"\n{'='*60}")
    print(f" Member {member_id + 1}/{N_MEMBERS}   (seed={seed})")
    print(f"{'='*60}")

    # Fresh model, optimizer, scheduler, criterion, early stopping
    model = GridInertiaPINN(
        input_dim  = INPUT_DIM,
        hidden_dim = HIDDEN_DIM,
        n_layers   = N_LAYERS,
        dropout_p  = DROPOUT_P,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=LR_FACTOR,
        patience=LR_PATIENCE, min_lr=LR_MIN, verbose=False,
    )
    criterion = PINNLoss(
        lambda_phys   = LAMBDA_PHYS,
        lambda_smooth = LAMBDA_SMOOTH,
        phys_warmup   = True,
        warmup_epochs = WARMUP_EPOCHS,
    )
    stopper = EarlyStopping(patience=ES_PATIENCE)

    member_log = []

    for epoch in range(1, EPOCHS + 1):
        train_bd = run_epoch(model, train_loader, criterion, optimizer, training=True)
        val_bd   = run_epoch(model, val_loader,   criterion, optimizer=None, training=False)

        scheduler.step(val_bd["loss_total"])
        criterion.step()
        stopper.step(val_bd["loss_total"], model)

        current_lr = optimizer.param_groups[0]["lr"]

        log_row = {
            "member":       member_id,
            "epoch":        epoch,
            "lr":           current_lr,
            "lambda_phys":  criterion.lambda_phys,
            **{f"train_{k}": v for k, v in train_bd.items()},
            **{f"val_{k}":   v for k, v in val_bd.items()},
        }
        member_log.append(log_row)

        if epoch % 10 == 0 or stopper.should_stop:
            print(
                f"  Ep {epoch:3d} | "
                f"train {train_bd['loss_total']:.4f} "
                f"(data {train_bd['loss_data']:.4f} "
                f"phys {train_bd['loss_swing']:.4f}) | "
                f"val {val_bd['loss_total']:.4f} | "
                f"lr {current_lr:.1e} | "
                f"λ_phys {criterion.lambda_phys:.3f}"
            )

        if stopper.should_stop:
            print(f"  Early stop at epoch {epoch}. Best val: {stopper.best_loss:.4f}")
            break

    # Restore best checkpoint and save
    stopper.restore_best(model)
    torch.save(model.state_dict(), CKPT_DIR / f"member_{member_id}.pt")
    all_logs.extend(member_log)
    print(f"  Saved → checkpoints/member_{member_id}.pt")


# Persist full training log
log_df = pd.DataFrame(all_logs)
log_df.to_csv(CKPT_DIR / "training_log.csv", index=False)
print(f"\nTraining log saved ({len(log_df)} rows).")

## 8. Training curves — all members

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

palette = sns.color_palette("tab10", N_MEMBERS)

for mid in range(N_MEMBERS):
    m_log = log_df[log_df["member"] == mid]
    c     = palette[mid]
    label = f"M{mid+1}"

    axes[0].plot(m_log["epoch"], m_log["train_loss_total"], color=c, lw=1.2, label=label)
    axes[0].plot(m_log["epoch"], m_log["val_loss_total"],   color=c, lw=1.2, linestyle="--")

    axes[1].plot(m_log["epoch"], m_log["train_loss_data"],  color=c, lw=1.2, label=label)
    axes[1].plot(m_log["epoch"], m_log["val_loss_data"],    color=c, lw=1.2, linestyle="--")

    axes[2].plot(m_log["epoch"], m_log["train_loss_swing"], color=c, lw=1.2, label=label)

for ax, title in zip(axes, ["Total loss", "Data loss (Huber)", "Physics loss (swing)"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Loss")
fig.suptitle("Training curves — solid: train, dashed: val", fontsize=11)
plt.tight_layout()
plt.show()

## 9. Load ensemble & run inference on test set

In [ ]:
# Rebuild ensemble and load saved weights
ensemble = DeepEnsemble(
    n_members  = N_MEMBERS,
    input_dim  = INPUT_DIM,
    hidden_dim = HIDDEN_DIM,
    n_layers   = N_LAYERS,
    dropout_p  = DROPOUT_P,
).to(DEVICE)
ensemble.load(CKPT_DIR)

# Full test-set inference (no windowing — pass every row)
ensemble.eval()
with torch.no_grad():
    X_test_t = torch.from_numpy(X_test).to(DEVICE)     # (N_test, 14)
    mean_pred, std_pred = ensemble(X_test_t)            # (3, N_test) each

H_mean  = mean_pred[0].cpu().numpy()    # H_sys mean
H_std   = std_pred[0].cpu().numpy()     # H_sys uncertainty
D_mean  = mean_pred[1].cpu().numpy()
D_std   = std_pred[1].cpu().numpy()
R_mean  = mean_pred[2].cpu().numpy()    # RoCoF mean
R_std   = std_pred[2].cpu().numpy()

print(f"Test samples   : {len(H_mean):,}")
print(f"H_sys pred     : mean={H_mean.mean():.3f}  std={H_mean.std():.3f}")
print(f"H_sys target   : mean={y_test.mean():.3f}  std={y_test.std():.3f}")
print(f"H_sys uncert.  : mean ensemble std = {H_std.mean():.4f}")

## 10. Evaluation metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mask_test = ~np.isnan(y_test)
y_true = y_test[mask_test]
y_pred = H_mean[mask_test]

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-6))) * 100

print("Test set metrics — H_sys prediction")
print(f"  MAE  : {mae:.4f} MWs/MVA")
print(f"  RMSE : {rmse:.4f} MWs/MVA")
print(f"  R²   : {r2:.4f}")
print(f"  MAPE : {mape:.2f}%")

metrics_df = pd.DataFrame([{"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}])
metrics_df.to_csv(CKPT_DIR / "test_metrics.csv", index=False)
print("Saved test_metrics.csv")

## 11. Prediction vs target — time series

In [ ]:
test_index = df_test.index
plot_slice = slice(0, min(4 * 96, len(test_index)))   # first 4 days

fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

# H_sys
axes[0].plot(test_index[plot_slice], y_test[plot_slice],
             color="black", lw=1, label="H_sys target (proxy)", zorder=3)
axes[0].plot(test_index[plot_slice], H_mean[plot_slice],
             color="steelblue", lw=1.2, label="H_sys predicted (ensemble mean)")
axes[0].fill_between(
    test_index[plot_slice],
    H_mean[plot_slice] - 2 * H_std[plot_slice],
    H_mean[plot_slice] + 2 * H_std[plot_slice],
    alpha=0.25, color="steelblue", label="±2σ ensemble uncertainty"
)
axes[0].set_ylabel("H_sys (MWs/MVA)")
axes[0].set_title(f"H_sys prediction — test set (R²={r2:.3f}, RMSE={rmse:.4f})")
axes[0].legend(fontsize=9)

# D(t)
axes[1].plot(test_index[plot_slice], D_mean[plot_slice],
             color="darkorange", lw=1.2, label="D(t) predicted")
axes[1].fill_between(
    test_index[plot_slice],
    D_mean[plot_slice] - 2 * D_std[plot_slice],
    D_mean[plot_slice] + 2 * D_std[plot_slice],
    alpha=0.25, color="darkorange"
)
axes[1].set_ylabel("D(t) (MW/Hz)")
axes[1].set_title("Damping coefficient D(t)")
axes[1].legend(fontsize=9)

# RoCoF
axes[2].plot(test_index[plot_slice], R_mean[plot_slice],
             color="crimson", lw=1, label="RoCoF predicted")
axes[2].fill_between(
    test_index[plot_slice],
    R_mean[plot_slice] - 2 * R_std[plot_slice],
    R_mean[plot_slice] + 2 * R_std[plot_slice],
    alpha=0.2, color="crimson"
)
axes[2].axhline(0, color="black", lw=0.8, linestyle="--")
axes[2].set_ylabel("RoCoF (Hz/s)")
axes[2].set_title("Rate of Change of Frequency (RoCoF)")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 12. Parity plot & residuals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Parity plot
lims = [min(y_true.min(), y_pred.min()) - 0.1,
        max(y_true.max(), y_pred.max()) + 0.1]
axes[0].scatter(y_true, y_pred, alpha=0.05, s=3, color="steelblue")
axes[0].plot(lims, lims, "r--", lw=1.5, label="Perfect fit")
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel("H_sys target"); axes[0].set_ylabel("H_sys predicted")
axes[0].set_title(f"Parity plot  (R²={r2:.3f})")
axes[0].legend()

# Residual distribution
residuals = y_pred - y_true
axes[1].hist(residuals, bins=80, color="steelblue", edgecolor="white", density=True)
axes[1].axvline(0, color="red", lw=1.5, linestyle="--")
axes[1].set_xlabel("Residual (predicted − target)")
axes[1].set_title(f"Residual distribution   μ={residuals.mean():.4f}  σ={residuals.std():.4f}")

# Uncertainty vs absolute error
abs_err = np.abs(residuals)
axes[2].scatter(H_std[mask_test], abs_err, alpha=0.05, s=3, color="darkorange")
axes[2].set_xlabel("Ensemble std (uncertainty)")
axes[2].set_ylabel("|Residual|")
axes[2].set_title("Uncertainty calibration")

plt.tight_layout()
plt.show()

## 13. Physics residual analysis

In [ ]:
from src.losses import SwingLoss

F0  = 50.0
EPS = 1e-6

dP_test_t = torch.from_numpy(dP_test)
Pt_test_t = torch.from_numpy(Pt_test)
H_pred_t  = torch.from_numpy(H_mean)
R_pred_t  = torch.from_numpy(R_mean)

swing_fn   = SwingLoss(weighted=False)
swing_res  = swing_fn.residuals(H_pred_t, R_pred_t, dP_test_t, Pt_test_t).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(swing_res, bins=100, color="crimson", edgecolor="white", density=True)
axes[0].axvline(0, color="black", lw=1.5, linestyle="--")
axes[0].set_xlabel("Swing equation residual (Hz/s)")
axes[0].set_title(
    f"Physics residual distribution\n"
    f"μ={swing_res.mean():.5f}  σ={swing_res.std():.5f}"
)

axes[1].plot(test_index[:4*96], np.abs(swing_res[:4*96]), color="crimson", lw=0.7)
axes[1].set_ylabel("|Swing residual|  (Hz/s)")
axes[1].set_title("Physics residual over time (first 4 days of test set)")

plt.tight_layout()
plt.show()

print(f"Mean |swing residual| : {np.abs(swing_res).mean():.6f} Hz/s")

## 14. Save full test predictions

In [ ]:
df_predictions = pd.DataFrame({
    "H_sys_target":   y_test,
    "H_sys_mean":     H_mean,
    "H_sys_std":      H_std,
    "D_mean":         D_mean,
    "D_std":          D_std,
    "RoCoF_mean":     R_mean,
    "RoCoF_std":      R_std,
    "swing_residual": swing_res,
    "delta_P":        dP_test,
    "P_total":        Pt_test,
}, index=df_test.index)

out_path = Path("../data/processed/de_predictions.csv")
df_predictions.to_csv(out_path)
print(f"Saved: {out_path}  ({df_predictions.shape})")
df_predictions.describe().round(4)

## 15. Summary

### What was trained
- **5-member Deep Ensemble** of `GridInertiaPINN` models, each independently seeded.
- Physics constraint: swing equation enforced via `SwingLoss` with warm-up schedule.
- Uncertainty: ensemble disagreement across members gives calibrated `H_sys` error bars.

### Outputs
| File | Contents |
|---|---|
| `checkpoints/member_{0..4}.pt` | Trained model weights |
| `checkpoints/scaler.pkl` | Feature normalisation (required for inference) |
| `checkpoints/training_log.csv` | Per-epoch loss breakdown for all members |
| `checkpoints/test_metrics.csv` | MAE, RMSE, R², MAPE on test set |
| `data/processed/de_predictions.csv` | Full test-set predictions + uncertainty |

### Next step → Notebook 04: Insights
`de_predictions.csv` feeds directly into the vulnerability analysis:
- Rolling Jitter Index `J(t) = Var(RoCoF) / H_sys`
- Low-inertia event detection and characterisation
- Monte Carlo demand shock simulations
- Publication-quality figures